In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model_provider="ollama",
    model="qwen3:8b",
                       )

In [3]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [4]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

<think>
Okay, the user has leftover chicken and rice and wants recipe ideas. Let me look through the search results to find the best options.

First, the Kitchn article mentions several dishes like Chicken and Rice Casserole, French onion chicken and rice, Italian chicken and rice, and a Greek version. There's also a creamy soup and a one-pan dish. These seem like good options. The Simply Recipes site has a baked chicken and rice recipe with lemon parsley, which sounds refreshing. Another entry from Simply Recipes talks about a dump-and-bake casserole with hot honey, which might be easy. The Reddit post mentions Hainanese chicken rice, which is a specific dish with boiled chicken and rice, maybe with some spices.

I should highlight a few varied recipes: a casserole, a one-pan meal, a soup, and maybe the Hainanese style. The user might prefer something quick or something more elaborate. Since they have leftovers, a casserole or one-pan dish would use the ingredients well. I'll list the

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken and rice. What can I make?', additional_kwargs={}, response_metadata={}, id='38304641-5794-441f-bd14-6a1c1f8f19ff'),
              AIMessage(content='<think>\nOkay, the user has leftover chicken and rice and wants to know what they can make. Let me think about possible recipes that use these ingredients. Chicken and rice are common components in many dishes. Maybe they can make a stir-fry, a casserole, or even a rice bowl. But I should verify if there are specific recipes that combine these two.\n\nFirst, I should use the web_search function to find recipes that include both chicken and rice. The search query should be something like "recipes with chicken and rice" to get relevant results. Once I get the search results, I can look for popular or simple recipes that might be easy to prepare. \n\nPossible options could be Chicken and Rice Casserole, Chicken Rice Bowl, or even a Chicken Rice Stir-Fry. Depending on the user\'